## 掩码自注意力机制


In [2]:
import torch
from torch import nn


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, attn_dim: int, output_dim: int, num_heads: int,):
        super().__init__()
        self.embed_dim = embed_dim
        self.attn_dim = attn_dim
        self.output_dim = output_dim
        self.num_heads = num_heads
        self.head_dim = attn_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)

        self.out_proj = nn.Linear(self.attn_dim, self.output_dim, bias=False)

    def forward(self, x,mask=None):
        batch_size, seq_len, embed_dim = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_score = torch.matmul(q, k.transpose(-2, -1))
        q_k = k.size(-1)
        attn_score = attn_score / torch.sqrt(torch.tensor(q_k))

        if mask is not None:
            attn_score = attn_score.masked_fill(mask == 0,float('-inf'))

        attn_weight = torch.softmax(attn_score, dim=-1)

        o = torch.matmul(attn_weight, v)
        attn_out = o.transpose(1, 2).reshape(batch_size, seq_len, self.attn_dim)
        return self.out_proj(attn_out)


In [4]:
x=torch.randn(1,4,2)
mask=torch.tril(torch.ones(4,4))
attn=MultiHeadAttention(2,6,8,2)
attn(x)

tensor([[[ 0.5853,  0.0072, -0.2972,  0.2107,  0.4700,  0.3333, -0.0761,
          -0.0822],
         [ 0.2704,  0.0611, -0.2715,  0.2122,  0.1217,  0.3173,  0.0134,
          -0.2538],
         [ 0.6834,  0.0043, -0.2888,  0.1601,  0.6371,  0.3104, -0.0985,
          -0.0613],
         [ 0.4298,  0.0463, -0.2744,  0.1758,  0.3422,  0.3093, -0.0265,
          -0.1973]]], grad_fn=<UnsafeViewBackward0>)